# Data exploration

> These are Go notebooks: In order to use the GoNB Jupyter Kernel, please install GoNB from here: https://github.com/janpfeifer/gonb

Note also that for local package development, you can put: `!*go mod edit -replace "github.com/umbralcalc/anglersim=/path/to/anglersim"` at the top of any cell.

In [ ]:
import (
	"os"
	"github.com/umbralcalc/anglersim/pkg/data"
)

%%

sdf := data.GetUniqueSitesDataFrameFromCountsCSV("../dat/FW_Fish_Counts.csv")
fmt.Println(sdf)

f, err := os.Create("../dat/FW_Fish_Unique_Count_Sites.csv")
if err != nil {
	panic(err)
}
sdf.WriteCSV(f)

In [ ]:
import (
	"gonum.org/v1/gonum/floats"
	"github.com/umbralcalc/anglersim/pkg/data"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

siteName := "Haxted Mill"
fdf := data.GetSiteCountsDataFrameFromCSV("../dat/FW_Fish_Counts.csv", siteName)

scatter := analysis.NewScatterPlotFromDataFrame(
	&fdf,
	"TIMESTAMP",
	"ALL_RUNS",
	"SPECIES_NAME",
)

xAxis := fdf.Col("TIMESTAMP").Float()
yAxis := fdf.Col("ALL_RUNS").Float()
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title: "Site: " + siteName,
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{
		Min: floats.Min(yAxis),
		Max: floats.Max(yAxis),
	}),
	charts.WithXAxisOpts(opts.XAxis{
		Min: floats.Min(xAxis),
		Max: floats.Max(xAxis),
	}),
)

gonb_echarts.Display(scatter, "width: 1024px; height:400px; background: white;")

In [ ]:
import (
    "github.com/umbralcalc/anglersim/pkg/data"
)

%%

df := data.GetTypesDataFrameFromCSV("../dat/FW_Fish_Data_Types.csv")

fmt.Println(df)

In [ ]:
import (
    "github.com/umbralcalc/anglersim/pkg/data"
)

%%

df := data.GetSitesDataFrameFromCSV("../dat/FW_Fish_Sites.csv")

fmt.Println(df)

In [ ]:
import (
    "bufio"
	"bytes"
	"fmt"
	"os"

	"github.com/go-gota/gota/dataframe"
)

%%

const maxRows = 100

f, err := os.Open("../dat/FW_Fish_Individual_Lengths.csv")
if err != nil {
	panic(err)
}

scanner := bufio.NewScanner(f)
var buffer bytes.Buffer

if scanner.Scan() {
	buffer.WriteString(scanner.Text() + "\n")
} else {
	panic("Empty file")
}

count := 0
for scanner.Scan() {
	buffer.WriteString(scanner.Text() + "\n")
	count++
	if count >= maxRows {
		break
	}
}
if err := scanner.Err(); err != nil {
	panic(err)
}

df := dataframe.ReadCSV(bytes.NewReader(buffer.Bytes()))

fmt.Println(df)

In [ ]:
import (
    "os"
	"github.com/go-gota/gota/dataframe"
)

%%

f, err := os.Open("../dat/FW_Fish_Banded_Measurements.csv")
if err != nil {
	panic(err)
}

df := dataframe.ReadCSV(f)

fmt.Println(df)

In [ ]:
import (
    "os"
	"github.com/go-gota/gota/dataframe"
)

%%

f, err := os.Open("../dat/FW_Fish_Bulk_Measurements.csv")
if err != nil {
	panic(err)
}

df := dataframe.ReadCSV(f)

fmt.Println(df)

# Data Coverage Analysis

How much data do we actually have? Can we fit time-series population models at individual sites, or do we need to pool across sites?

In [ ]:
import (
	"github.com/umbralcalc/anglersim/pkg/data"
)

%%

report := data.GetCoverageReport("../dat/FW_Fish_Counts.csv")
report.PrintSummary()

In [ ]:
import (
	"gonum.org/v1/gonum/floats"
	"github.com/umbralcalc/anglersim/pkg/data"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// Records per year — shows survey effort over time
report := data.GetCoverageReport("../dat/FW_Fish_Counts.csv")
yearDF := report.YearCountsDataFrame()

scatter := analysis.NewScatterPlotFromDataFrame(&yearDF, "YEAR", "RECORDS")
xAxis := yearDF.Col("YEAR").Float()
yAxis := yearDF.Col("RECORDS").Float()
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title: "NFPD Survey Records Per Year",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{
		Min: floats.Min(yAxis),
		Max: floats.Max(yAxis),
	}),
	charts.WithXAxisOpts(opts.XAxis{
		Min: floats.Min(xAxis),
		Max: floats.Max(xAxis),
	}),
)
gonb_echarts.Display(scatter, "width: 1024px; height:400px; background: white;")

In [ ]:
import (
	"github.com/umbralcalc/anglersim/pkg/data"
)

%%

// Top 20 species by number of records
report := data.GetCoverageReport("../dat/FW_Fish_Counts.csv")
sppDF := report.SpeciesCountsDataFrame()
fmt.Println(sppDF.Subset([]int{0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19}))

In [ ]:
import (
	"github.com/umbralcalc/anglersim/pkg/data"
)

%%

// Sites with the longest time series (≥10 distinct survey years)
report := data.GetCoverageReport("../dat/FW_Fish_Counts.csv")
coverageDF := report.SiteCoverageDataFrame()
longSeries := report.SitesWithMinYears(10)
fmt.Printf("Sites with ≥10 survey years: %d\n", len(longSeries))
fmt.Printf("Sites with ≥5 survey years:  %d\n", len(report.SitesWithMinYears(5)))
fmt.Printf("Sites with ≥15 survey years: %d\n", len(report.SitesWithMinYears(15)))
fmt.Printf("Sites with ≥20 survey years: %d\n\n", len(report.SitesWithMinYears(20)))

// Show top 20 sites
fmt.Println("Top 20 sites by number of survey years:")
fmt.Println(coverageDF.Subset([]int{0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19}))

## Example: Long time-series site

Let's look at one of the best-covered sites to see what the data looks like for population modelling. We want to check: are surveys roughly annual? Are species consistently recorded? Is there enough signal to fit a stochastic model?

In [ ]:
import (
	"gonum.org/v1/gonum/floats"
	"github.com/umbralcalc/anglersim/pkg/data"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// Mill Meadow — 31 survey years, 20 species, 45-year span
siteName := "Mill Meadow"
fdf := data.GetSiteCountsDataFrameFromCSV("../dat/FW_Fish_Counts.csv", siteName)

scatter := analysis.NewScatterPlotFromDataFrame(
	&fdf,
	"TIMESTAMP",
	"ALL_RUNS",
	"SPECIES_NAME",
)

xAxis := fdf.Col("TIMESTAMP").Float()
yAxis := fdf.Col("ALL_RUNS").Float()
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title: "Site: " + siteName + " (31 survey years, 20 species)",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{
		Min: floats.Min(yAxis),
		Max: floats.Max(yAxis),
	}),
	charts.WithXAxisOpts(opts.XAxis{
		Min: floats.Min(xAxis),
		Max: floats.Max(xAxis),
	}),
)

gonb_echarts.Display(scatter, "width: 1024px; height:400px; background: white;")

# Prepared Panel Data: Brown Trout

Build a clean site x year panel for brown/sea trout using only electrofishing surveys, filtering to sites with 10+ survey years. This handles:
- Multiple surveys per site-year (summed)
- Zero-catch detection (site surveyed but species absent)
- Density calculation (count / fished area)

In [ ]:
import (
	"github.com/umbralcalc/anglersim/pkg/data"
)

%%

cfg := data.PanelConfig{
	TargetSpecies:      "Brown / sea trout",
	MinSurveyYears:     10,
	ElectrofishingOnly: true,
}
panel := data.BuildPanel("../dat/FW_Fish_Counts.csv", cfg)

fmt.Printf("Panel: %d site-year records\n\n", len(panel))

summary := data.PanelSiteSummary(panel)
fmt.Printf("Sites with ≥10 survey years: %d\n\n", summary.Nrow())
fmt.Println("Top 15 sites by number of survey years:")
fmt.Println(summary.Subset([]int{0,1,2,3,4,5,6,7,8,9,10,11,12,13,14}))

In [ ]:
import (
	"gonum.org/v1/gonum/floats"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	"github.com/umbralcalc/anglersim/pkg/data"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// Build panel and plot density time series for one of the best sites
cfg := data.PanelConfig{
	TargetSpecies:      "Brown / sea trout",
	MinSurveyYears:     10,
	ElectrofishingOnly: true,
}
panel := data.BuildPanel("../dat/FW_Fish_Counts.csv", cfg)
panelDF := data.PanelToDataFrame(panel)

siteName := "Sydenham"
siteDF := panelDF.Filter(dataframe.F{
	Colname:    "SITE_NAME",
	Comparator: series.Eq,
	Comparando: siteName,
})

scatter := analysis.NewScatterPlotFromDataFrame(&siteDF, "YEAR", "DENSITY")
xAxis := siteDF.Col("YEAR").Float()
yAxis := siteDF.Col("DENSITY").Float()
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title: "Brown trout density: " + siteName + " (fish/m²)",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{
		Min: 0.0,
		Max: floats.Max(yAxis) * 1.1,
	}),
	charts.WithXAxisOpts(opts.XAxis{
		Min: floats.Min(xAxis) - 1,
		Max: floats.Max(xAxis) + 1,
	}),
)
gonb_echarts.Display(scatter, "width: 1024px; height:400px; background: white;")

In [ ]:
import (
	"os"
	"github.com/umbralcalc/anglersim/pkg/data"
)

%%

// Build panel and save to CSV for downstream use
cfg := data.PanelConfig{
	TargetSpecies:      "Brown / sea trout",
	MinSurveyYears:     10,
	ElectrofishingOnly: true,
}
panel := data.BuildPanel("../dat/FW_Fish_Counts.csv", cfg)
panelDF := data.PanelToDataFrame(panel)
summary := data.PanelSiteSummary(panel)

f, err := os.Create("../dat/brown_trout_panel.csv")
if err != nil {
	panic(err)
}
panelDF.WriteCSV(f)
f.Close()

sf, err := os.Create("../dat/brown_trout_sites.csv")
if err != nil {
	panic(err)
}
summary.WriteCSV(sf)
sf.Close()

fmt.Println("Saved dat/brown_trout_panel.csv and dat/brown_trout_sites.csv")

# Hydrology & Water Quality Data

Fetch environmental covariates for brown trout panel sites from EA APIs:
- **Hydrology API** (`environment.data.gov.uk/hydrology/`): Daily mean river flow (m³/s)
- **Water Quality API** (`environment.data.gov.uk/water-quality/`): Temperature, dissolved oxygen, ammonia, BOD

Each panel site is matched to the nearest monitoring station/sampling point within a search radius.

In [ ]:
import (
	"fmt"
	"github.com/umbralcalc/anglersim/pkg/data"
)

%%

// Find the nearest flow station to the best brown trout site (Sydenham)
// Sydenham easting/northing from panel output
easting := 242977
northing := 83838

station := data.FindNearestFlowStation(easting, northing, 10.0)
if station == nil {
	fmt.Println("No flow station found within 10km")
} else {
	fmt.Printf("Station: %s\n", station.Label)
	fmt.Printf("Easting: %d, Northing: %d\n", station.Easting, station.Northing)
	fmt.Printf("Lat: %.4f, Long: %.4f\n", station.Lat, station.Long)
	fmt.Printf("Date opened: %s\n", station.DateOpened)
	fmt.Printf("Daily mean flow measure: %s\n", station.DailyMeanFlowMeasureID())
	fmt.Printf("Number of measures: %d\n", len(station.Measures))
	for _, m := range station.Measures {
		fmt.Printf("  - %s (period=%ds)\n", m.Parameter, m.Period)
	}
}

In [ ]:
import (
	"fmt"
	"gonum.org/v1/gonum/floats"
	"github.com/umbralcalc/anglersim/pkg/data"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// Fetch daily mean flow readings for a station near Sydenham and plot annual stats
station := data.FindNearestFlowStation(242977, 83838, 10.0)
if station == nil {
	panic("No flow station found")
}

measureID := station.DailyMeanFlowMeasureID()
fmt.Printf("Fetching flow readings for %s (measure: %s)...\n", station.Label, measureID)

readings := data.GetDailyFlowReadings(measureID, "1984-01-01", "2025-01-01")
fmt.Printf("Got %d daily readings\n\n", len(readings))

annualStats := data.AggregateAnnualFlow(readings)
flowDF := data.AnnualFlowToDataFrame(annualStats)
fmt.Println(flowDF)

// Plot mean annual flow
scatter := analysis.NewScatterPlotFromDataFrame(&flowDF, "YEAR", "MEAN_FLOW")
xAxis := flowDF.Col("YEAR").Float()
yAxis := flowDF.Col("MEAN_FLOW").Float()
scatter.SetGlobalOptions(
	charts.WithTitleOpts(opts.Title{
		Title: "Mean Annual Flow: " + station.Label + " (m³/s)",
		Bottom: "1%",
	}),
	charts.WithYAxisOpts(opts.YAxis{
		Min: 0.0,
		Max: floats.Max(yAxis) * 1.1,
	}),
	charts.WithXAxisOpts(opts.XAxis{
		Min: floats.Min(xAxis) - 1,
		Max: floats.Max(xAxis) + 1,
	}),
)
gonb_echarts.Display(scatter, "width: 1024px; height:400px; background: white;")

In [ ]:
import (
	"fmt"
	"github.com/umbralcalc/anglersim/pkg/data"
)

%%

// Find the nearest water quality sampling point to Sydenham
wqPt := data.FindNearestWQPoint(242977, 83838, 10.0)
if wqPt == nil {
	fmt.Println("No water quality sampling point found within 10km")
} else {
	fmt.Printf("Sampling point: %s\n", wqPt.Name)
	fmt.Printf("Notation: %s\n", wqPt.Notation)
	fmt.Printf("Type: %s, Status: %s\n", wqPt.Type, wqPt.Status)
	fmt.Printf("Lat: %.4f, Long: %.4f\n", wqPt.Lat, wqPt.Long)
	fmt.Printf("Distance: %.2f km\n", wqPt.DistKm)
}

In [ ]:
import (
	"fmt"
	"gonum.org/v1/gonum/floats"
	"github.com/umbralcalc/anglersim/pkg/data"
	"github.com/umbralcalc/stochadex/pkg/analysis"
	gonb_echarts "github.com/janpfeifer/gonb-echarts"
)

%%

// Fetch water quality observations and plot annual temperature
wqPt := data.FindNearestWQPoint(242977, 83838, 10.0)
if wqPt == nil {
	panic("No WQ sampling point found")
}

fmt.Printf("Fetching water quality for %s (%s, %.1f km away)...\n", wqPt.Name, wqPt.Notation, wqPt.DistKm)

tempObs := data.GetWQObservations(wqPt.Notation, data.DetTemperature, "1984-01-01", "2025-01-01")
doObs := data.GetWQObservations(wqPt.Notation, data.DetDissolvedO2, "1984-01-01", "2025-01-01")
nh3Obs := data.GetWQObservations(wqPt.Notation, data.DetAmmonia, "1984-01-01", "2025-01-01")
bodObs := data.GetWQObservations(wqPt.Notation, data.DetBOD, "1984-01-01", "2025-01-01")

fmt.Printf("Observations: temp=%d, DO=%d, NH3=%d, BOD=%d\n\n",
	len(tempObs), len(doObs), len(nh3Obs), len(bodObs))

annualWQ := data.AggregateAnnualWQ(tempObs, doObs, nh3Obs, bodObs)
wqDF := data.AnnualWQToDataFrame(annualWQ)
fmt.Println(wqDF)

// Plot mean annual temperature
scatter := analysis.NewScatterPlotFromDataFrame(&wqDF, "YEAR", "MEAN_TEMP")
xAxis := wqDF.Col("YEAR").Float()
yAxis := wqDF.Col("MEAN_TEMP").Float()
validY := make([]float64, 0)
for _, v := range yAxis {
	if v == v { // skip NaN
		validY = append(validY, v)
	}
}
if len(validY) > 0 {
	scatter.SetGlobalOptions(
		charts.WithTitleOpts(opts.Title{
			Title: "Mean Annual Water Temp: " + wqPt.Name + " (°C)",
			Bottom: "1%",
		}),
		charts.WithYAxisOpts(opts.YAxis{
			Min: floats.Min(validY) * 0.9,
			Max: floats.Max(validY) * 1.1,
		}),
		charts.WithXAxisOpts(opts.XAxis{
			Min: floats.Min(xAxis) - 1,
			Max: floats.Max(xAxis) + 1,
		}),
	)
	gonb_echarts.Display(scatter, "width: 1024px; height:400px; background: white;")
} else {
	fmt.Println("No temperature data available for plotting")
}

In [ ]:
import (
	"fmt"
	"github.com/go-gota/gota/dataframe"
	"github.com/go-gota/gota/series"
	"github.com/umbralcalc/anglersim/pkg/data"
)

%%

// Match a subset of panel sites to flow stations and WQ sampling points
// (Use a small subset first to test — full matching takes many API calls)
cfg := data.PanelConfig{
	TargetSpecies:      "Brown / sea trout",
	MinSurveyYears:     20, // top sites only for quick test
	ElectrofishingOnly: true,
}
panel := data.BuildPanel("../dat/FW_Fish_Counts.csv", cfg)
summary := data.PanelSiteSummary(panel)

// Take top 10 sites
topN := 10
if summary.Nrow() < topN {
	topN = summary.Nrow()
}
indices := make([]int, topN)
for i := range indices {
	indices[i] = i
}
topSites := summary.Subset(indices)

fmt.Printf("Matching %d sites to flow stations...\n", topN)
flowMatch := data.MatchPanelSitesToFlowStations(topSites, 10.0)
flowMatched := flowMatch.Filter(dataframe.F{Colname: "MATCHED", Comparator: series.Eq, Comparando: true})
fmt.Printf("Flow stations matched: %d/%d\n\n", flowMatched.Nrow(), topN)
fmt.Println(flowMatch)

fmt.Printf("\nMatching %d sites to WQ sampling points...\n", topN)
wqMatch := data.MatchPanelSitesToWQPoints(topSites, 10.0)
wqMatched := wqMatch.Filter(dataframe.F{Colname: "MATCHED", Comparator: series.Eq, Comparando: true})
fmt.Printf("WQ points matched: %d/%d\n\n", wqMatched.Nrow(), topN)
fmt.Println(wqMatch)